# Sentinel: adding new recordings

Turns new raw recordings (`.jsonl` from `scenarios/raw_recorder.py`) into feature rows with
`sentinel/features.py`, checks them, and either **adds them to the training table**
(`features.csv`) or stores them as an **extra test set** that the model never trains on
(`features_busy.csv`, used in section 8.5 of `train_sentinel.ipynb`).

Afterwards, rerun `train_sentinel.ipynb` (Run all).

**History of the training data**

| Date | Change |
| --- | --- |
| 9 Nov | 30 runs recorded (S1-S7). S2_run2 had no label line; labelled normal from its own timestamps. |
| 17 Nov | The 9 Nov S2 runs turned out to contain no iperf traffic: renamed S0_idle_run1/2. Four runs with working iperf added as normal traffic, named in features.csv as S2_iperf_run1/2 and S3_iperf_run1/2 (in labels.jsonl: S2_20261117_022834, S2_20261117_091134, S3_20261117_092939, S3_20261117_094800).
| 17 Nov | Four attack runs during busy traffic (S5, S6) stored as the extra test set `features_busy.csv`. |

## 1. Paths

In [ ]:
import os, sys, json, glob, shutil, subprocess
from pathlib import Path
import pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/sdn-sentinel')
    CODE_DIR, FEATURES_CSV, BUSY_CSV = BASE, BASE / 'features.csv', BASE / 'features_busy.csv'
else:
    ROOT = Path('..').resolve()
    CODE_DIR, FEATURES_CSV, BUSY_CSV = ROOT, ROOT / 'data' / 'features.csv', ROOT / 'data' / 'features_busy.csv'
FEATURES_PY = CODE_DIR / 'sentinel' / 'features.py'
print('features.py :', FEATURES_PY, '(exists)' if FEATURES_PY.exists() else '(MISSING)')
print('train table :', FEATURES_CSV, '(exists)' if FEATURES_CSV.exists() else '(MISSING)')

## 2. What to add

* `NEW_DIR`: folder holding the new `.jsonl` recordings (unzip them there first).
* `TARGET`: `'train'` adds them to `features.csv`; `'extra_test'` writes them to `features_busy.csv` only.
* `NORMAL_RUNS`: for **normal** runs, map each file name (without `.jsonl`) to a new, unique run id;
  their label (normal for the whole run) is created from the recording's own timestamps.
* `LABELS_FILE`: for **attack** runs, the `labels.jsonl` from the recording machine, holding the exact
  attack start and end times. Lines are matched to recordings by time.

In [ ]:
NEW_DIR = (BASE if IN_COLAB else ROOT / 'data') / 'new_runs_2'
TARGET = 'extra_test'                      # 'train' or 'extra_test'
NORMAL_RUNS = {}                           # e.g. {'S2_run1': 'S2_iperf_run1'} for normal runs
LABELS_FILE = NEW_DIR / 'labels.jsonl'     # needed for attack runs

## 3. Check the recordings

In [ ]:
recs = sorted(p for p in glob.glob(f'{NEW_DIR}/**/*.jsonl', recursive=True) if 'label' not in os.path.basename(p))
labels = [json.loads(l) for l in open(LABELS_FILE) if l.strip()] if Path(LABELS_FILE).exists() else []
print(len(recs), 'recordings,', len(labels), 'label lines\n')
for path in recs:
    first = last = meta = None; n = 0; gaps = []; prev = None; max_flows = 0
    for line in open(path):
        if '"meta"' in line[:40]: meta = json.loads(line); continue
        if '"sample"' not in line[:40]: continue
        o = json.loads(line); n += 1; first = first or o; last = o
        max_flows = max(max_flows, len(o['flows']))
        if prev: gaps.append(o['ts'] - prev)
        prev = o['ts']
    core = {tuple(l[k].rsplit('/', 1)) for l in meta['links'] for k in ('src', 'dst')}
    start = {(p['dev'], str(p['port'])): p['rx'] for p in first['ports']}
    edge_rx = {f"{k[0][-1]}:{k[1]}": p['rx'] - start.get(k, 0) for p in last['ports']
               for k in [(p['dev'], str(p['port']))] if k not in core and str(p['port']).upper() != 'LOCAL'}
    name = os.path.basename(path)[:-6]
    match = [l for l in labels if l['t_start'] < last['ts'] and l['t_end'] > first['ts']]
    print(f"{name}: {n} snapshots, {(last['ts'] - first['ts']) / 60:.1f} min, step median "
          f"{sorted(gaps)[len(gaps) // 2]:.1f} s, max flow rules {max_flows}, hosts at end {len(last['hosts'])}")
    print('   packets per host port over the run:', edge_rx)
    if name in NORMAL_RUNS:
        print('   normal run, will be named', NORMAL_RUNS[name])
    elif match:
        for l in match:
            print(f"   label {l['run_id']} | {l['label']} | attacker {l['attacker']} | attack "
                  f"{l['t_start'] - first['ts']:.0f}-{l['t_end'] - first['ts']:.0f} s into the recording")
    else:
        print('   NO LABEL: add it to NORMAL_RUNS (normal run) or to LABELS_FILE (attack run)')

Check before going on: the step must be about **2.0 s** (the model's step); normal runs that are
meant to be busy should show **thousands or more** packets on the host ports; every attack run needs
a label line.

## 4. Build the features

In [ ]:
lab_path = Path('/tmp' if not IN_COLAB else '/content') / 'labels_for_new_runs.jsonl'
lines = [l for l in labels]
for fname, rid in NORMAL_RUNS.items():
    path = next(p for p in recs if os.path.basename(p)[:-6] == fname)
    ts = [json.loads(l)['ts'] for l in open(path) if '"sample"' in l[:40]]
    lines.append({'run_id': rid, 'scenario': rid[:2], 'label': 'normal', 'attacker': None,
                  't_start': ts[0], 't_end': ts[-1]})
lab_path.write_text('\n'.join(json.dumps(l) for l in lines) + '\n')
new_csv = lab_path.with_name('features_new_runs.csv')
out = subprocess.run([sys.executable, str(FEATURES_PY), *recs, '--labels', str(lab_path), '--out', str(new_csv)],
                     capture_output=True, text=True)
print(out.stderr)
new = pd.read_csv(new_csv)
print(new.groupby(['run_id', 'label']).size())

## 5. Add them

In [ ]:
def append(path, new, backup_suffix):
    if path.exists():
        old = pd.read_csv(path)
        dup = sorted(set(old.run_id) & set(new.run_id))
        if dup:
            print('already present, nothing added:', dup); return
        backup = path.with_name(path.stem + backup_suffix + '.csv')
        shutil.copy(path, backup)
        merged = pd.concat([old, new], ignore_index=True)
        note = f'previous version kept as {backup.name}'
    else:
        merged, note = new, 'new file'
    merged.to_csv(path, index=False)
    print(f'{path.name}: {merged.shape[0]} rows from {merged.run_id.nunique()} runs ({note})')

if TARGET == 'train':
    append(FEATURES_CSV, new, '_backup')
else:
    append(BUSY_CSV, new, '_backup')
print('Now rerun train_sentinel.ipynb (Run all).')